In [0]:
df = spark.read.csv("/Volumes/workspace/default/my_volume/WA_Fn-UseC_-Telco-Customer-Churn.csv",
header=True,
inferSchema=True,
sep=",")

In [0]:
display(df)

In [0]:
# Bronze layer: persist the raw data exactly as read from the volume
bronze_table = "workspace.default.telco_customer_churn_bronze"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(bronze_table)

print(f"Bronze table written: {bronze_table} ({df.count()} rows)")
display(spark.table(bronze_table))

In [0]:
# Silver layer: cleaned data — drop nulls and remove the Partner column
silver_table = "workspace.default.telco_customer_churn_silver"

bronze_df = spark.table("workspace.default.telco_customer_churn_bronze")

silver_df = (bronze_df
    .drop("Partner")
    .na.drop(subset=[c for c, t in bronze_df.dtypes if t == "string"]) if False else bronze_df.drop("Partner"))

# Drop rows containing any null value
silver_df = silver_df.na.drop()

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print(f"Silver table written: {silver_table} ({silver_df.count()} rows)")
display(spark.table(silver_table))

In [0]:
# Gold layer: compact churn summary per segment
from pyspark.sql import functions as F

silver_df = spark.table("workspace.default.telco_customer_churn_silver")
gold_table = "workspace.default.telco_customer_churn_gold"

# Bucket raw tenure months into readable lifecycle segments
silver_df = silver_df.withColumn(
    "tenure_bucket",
    F.when(F.col("tenure") <= 12, "0-12")
     .when(F.col("tenure") <= 24, "13-24")
     .when(F.col("tenure") <= 48, "25-48")
     .otherwise("49-72"),
)

agg_exprs = [
    F.count("*").alias("customers"),
    F.round(F.avg(F.when(F.col("Churn") == "Yes", 1.0).otherwise(0.0)) * 100, 2).alias("churn_rate_pct"),
    F.round(F.avg("MonthlyCharges"), 2).alias("avg_monthly_charges"),
]

# Core business segments: contract type, internet service, tenure, and age — the biggest churn drivers
segments = {
    "contract": "Contract",
    "internet_service": "InternetService",
    "tenure": "tenure_bucket",
    "age": "SeniorCitizen",
}

gold_df = None
for segment_name, col in segments.items():
    seg_df = (silver_df.groupBy(col).agg(*agg_exprs)
              .withColumn("segment_type", F.lit(segment_name))
              .withColumnRenamed(col, "segment_value")
              .withColumn("segment_value", F.col("segment_value").cast("string")))
    gold_df = seg_df if gold_df is None else gold_df.unionByName(seg_df)

gold_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_table)

print(f"Gold table written: {gold_table} ({gold_df.count()} rows)")
display(spark.table(gold_table))

# --- Elaborate the gold table output ---

# 1) Show the schema of the gold table
print("--- Gold table schema ---")
spark.table(gold_table).printSchema()

# 2) Show how many rows each segment type contributes
print("--- Rows per segment type ---")
display(
    spark.table(gold_table)
        .groupBy("segment_type")
        .agg(
            F.count("*").alias("segment_rows"),
            F.round(F.min("churn_rate_pct"), 2).alias("min_churn_rate_pct"),
            F.round(F.max("churn_rate_pct"), 2).alias("max_churn_rate_pct"),
        )
        .orderBy("segment_type")
)

# Output explanation:
# - segment_type   : which dimension the row aggregates (contract, internet_service, tenure, age)
# - segment_value  : the specific value within that dimension (e.g. 'Month-to-month', '0-12', '1')
# - customers      : number of customers in that segment
# - churn_rate_pct : % of those customers who churned (Churn == 'Yes')
# - avg_monthly_charges : average MonthlyCharges for the segment
# Each dimension's rows partition the full silver population, so segment rows
# within one segment_type sum back to the total customer count.